# <font color="steelblue">Severidad de cáncer de pulmón</font>

**Material desarrollado por los [equipos de trabajo de IA4LEGOS](https://ia4legos.umh.es/)**

**Licencia**: <a rel="license" href="http://creativecommons.org/licenses/by-sa/4.0/"><img alt="Creative Commons License" style="border-width:0" src="https://i.creativecommons.org/l/by-sa/4.0/88x31.png" /></a>

No olvides hacer una copia si deseas utilizarlo.

> **Guión de trabajo para el estudiante.** Este cuaderno es el **enunciado** de un proyecto de **aprendizaje no supervisado** que debéis **desarrollar vosotros**: objetivos, fases, tareas obligatorias, andamiaje mínimo de código  y criterios de evaluación. **No es una solución.**

## <font color="steelblue">Objetivos del proyecto</font>

Cada paciente está descrito por **20 factores de riesgo y síntomas** (escalas ordinales tratadas como **numéricas**) y tiene una **severidad** real `Level` (**Low < Medium < High**). **Ignoramos `Level` para agrupar** y luego **comparamos** la solución de cluster con esa etiqueta **ordinal**. Objetivos:

1. **Descubrir estructura** sin la etiqueta: reducción de dimensión (PCA/MDS) + **clustering** con varios algoritmos.
2. **Validar externamente** con métricas adecuadas a una etiqueta **ordinal de 3 niveles**: contingencia, ARI/NMI, mapeo cluster→nivel + `classification_report`, y **QWK/MAE/Spearman**.
3. **Pensar con escepticismo:** este dataset es sospechosamente "limpio" (en clasificación supervisada roza el 100 %). Si los clusters recuperan los 3 niveles **casi perfectamente**, valora si es un mérito del método o una **señal de que los datos podrían ser sintéticos/demasiado separables**.

> **Ordinal, no nominal:** como `Level` tiene orden, además de ARI/NMI usamos **QWK** (penaliza más confundir Low↔High que Medium↔High), **MAE** sobre códigos 0/1/2 y **Spearman**.

## <font color="steelblue">El conjunto de datos</font>

**1.001 pacientes**, **20 variables** predictoras + `Level` (The Devastator, 2022). Sin valores perdidos. La mayoría de variables son **escalas ordinales numéricas** (1–7, 1–8 o 1–9; a mayor valor, mayor intensidad/exposición) → se tratan como **numéricas**.

* **Demográficas:** `Age` (años, **escala muy distinta**), `Gender` (1/2).
* **Riesgo/hábitos:** `Air Pollution`, `Alcohol use`, `Dust Allergy`, `OccuPational Hazards`, `Genetic Risk`, `Balanced Diet`, `Obesity`, `Smoking`, `Passive Smoker`.
* **Síntomas/signos:** `chronic Lung Disease`, `Chest Pain`, `Coughing of Blood`, `Fatigue`, `Weight Loss`, `Shortness of Breath`, `Wheezing`, `Swallowing Difficulty`, `Clubbing of Finger Nails`.
* **Etiqueta `Level`:** severidad **ordinal** `Low` < `Medium` < `High`.

> **Carga (cambio importante):** el código original **elimina `Level`**; aquí la **CONSERVAMOS** como etiqueta de comparación. Eliminamos solo `index`, `Patient Id` y los síntomas `Frequent Cold`, `Dry Cough`, `Snoring` (siguiendo el preprocesado documentado de 20 variables).

## <font color="steelblue">Reglas del juego</font>

1. **Reserva `Level`**: el clustering se hace **solo** con las 20 predictoras. La etiqueta se usa **únicamente** en la comparación.
2. **Trata las ordinales como numéricas** (según el enunciado) y **escala**: `Age` tiene un rango enorme frente a las escalas 1–9.
3. **Trata la redundancia/colinealidad** (factores de riesgo correlacionados) y coméntala.
4. **Combina modelos:** reduce dimensión + agrupa; compara varios algoritmos.
5. **Elige el nº de grupos por criterios** y discútelo frente a los **3 niveles** conocidos.
6. **Comparación ORDINAL:** ARI/NMI + mapeo + `classification_report` **y** QWK/MAE/Spearman.
7. **Escepticismo:** una recuperación casi perfecta debe **levantar sospechas** sobre los datos, no celebrarse sin más. `random_state` fijado.

# <font color="steelblue">Fase 0 — Preparación del entorno y carga de datos</font>

In [ ]:
path = kagglehub.dataset_download("thedevastator/cancer-patients-and-air-pollution-a-new-link")
print("Ruta:", path, "| Archivos:", os.listdir(path))
lungcancer = pd.read_csv(os.path.join(path, "cancer patient data sets.csv"))

# CONSERVAMOS 'Level' (etiqueta de comparación). Eliminamos solo identificadores y 3 síntomas.
lungcancer = lungcancer.drop(columns=['index', 'Patient Id', 'Frequent Cold', 'Dry Cough', 'Snoring'])
print(f"Dimensiones: {lungcancer.shape[0]} filas × {lungcancer.shape[1]} columnas (incluye Level)")
lungcancer.head()

# <font color="steelblue">Fase 1 — Comprensión y EDA</font>

**Tareas a desarrollar**
1. **Etiqueta.** Distribución de `Level` (`Low`/`Medium`/`High`): es **ordinal**. (La reservaremos.)
2. **Predictoras ordinales-como-numéricas.** Revisa rangos (1–7/1–8/1–9) y, sobre todo, que `Age` está en **otra escala** → justifica el escalado.
3. **Correlaciones.** Muchos factores de riesgo y síntomas estarán **correlacionados**: localiza bloques.
4. **Anticipo visual.** PCA/t-SNE coloreado por `Level`: ¿se separan los niveles? (si la separación es **demasiado** nítida, anótalo: lo retomaremos con escepticismo).
5. **Conclusión:** 3–4 hallazgos.

# <font color="steelblue">Fase 2 — Preprocesado (reservando la etiqueta ordinal)</font>

**Tareas a desarrollar**
1. **Separa** `y = Level` como **ordinal ordenada** (`Low` < `Medium` < `High`) y resérvala; `X` = las 20 predictoras.
2. **Escala** `X` (StandardScaler): imprescindible por `Age`.
3. **(Opcional) Redundancia:** PCA/selección por la colinealidad; compara.

> **A responder:** ¿por qué el clustering **no** debe ver `Level`? ¿qué validez tendría la comparación si la usaras?

# <font color="steelblue">Fase 3 — Reducción de dimensión</font>

**Tareas a desarrollar**
1. **PCA** sobre `X` escalado: **scree plot** e **interpretación de loadings** (¿PC1 = "gradiente de severidad/exposición"?).
2. **Visualización.** PC1–PC2 y **MDS**/**t-SNE**, coloreando por `Level`.
3. **Conclusión:** dimensionalidad efectiva y qué capturan las componentes.

# <font color="steelblue">Fase 4 — Agrupación (clustering) y combinación de modelos</font>

**Tareas a desarrollar**
1. Aplica y compara **≥3** algoritmos: **K-Means**, **jerárquico** (+ dendrograma), **DBSCAN** y/o **GMM**.
2. **Combina con la reducción:** agrupa sobre el **espacio PCA** y compáralo con `X` escalado.
3. **Concordancia** entre soluciones (Adjusted Rand Index).

> Para comparar con 3 niveles puedes mirar `k=3`, pero **explóralo** en la Fase 5.

# <font color="steelblue">Fase 5 — Validación interna y número de grupos</font>

**Tareas a desarrollar**
1. **Número de grupos:** codo + **silueta** variando `k`; dendrograma.
2. **Índices internos:** silueta, **Davies–Bouldin**, **Calinski–Harabasz**.
3. **Estabilidad** ante semillas/submuestras.
4. **Discusión:** ¿el `k` óptimo interno **coincide con 3**? Si la estructura "natural" son exactamente 3 grupos muy separados, **anótalo** (lo cruzaremos con el escepticismo de la Fase 6/7).

# <font color="steelblue">Fase 6 — Comparación con la etiqueta ordinal (validación externa)</font>

La fase **distintiva**: contrastar la **solución de cluster** con la **severidad** `Level` (ordinal Low<Medium<High), con métricas **nominales y ordinales**.

**Tareas obligatorias**
1. **Matriz de contingencia** grupos × `Level` (`pd.crosstab`): léela como "confusión" y fíjate en la **distancia a la diagonal**.
2. **Índices externos nominales:** **ARI** y **NMI** (ignoran el orden: referencia parcial).
3. **Mapeo cluster→nivel** (clase mayoritaria) → `y_pred` ordinal y **`classification_report`** (macro-F1 ignora el orden).
4. **Métricas ORDINALES (clave):** con `y_true`/`y_pred` en códigos 0/1/2,
   * **QWK** — `cohen_kappa_score(..., weights='quadratic')`,
   * **MAE ordinal**,
   * **Spearman**.
5. **Escepticismo (clave).** Si la coincidencia es **casi perfecta** (ARI≈1, QWK≈1), pregúntate: ¿de verdad las 20 escalas separan tan limpiamente la severidad, o el dataset está **construido/sintetizado** para ser separable? Relaciónalo con el **~100 %** que da en clasificación supervisada. Propón una comprobación (p. ej. mirar si pocas variables ya separan perfectamente, o ruido en las fronteras).

# <font color="steelblue">Fase 7 — Caracterización de los grupos y lectura crítica</font>

**Tareas a desarrollar**
1. **Perfiles.** Media de cada variable (valores **originales**) por grupo → **mapa de calor**.
2. **¿Gradiente coherente?** ¿Los grupos asociados a `High` muestran de forma **consistente** más `Smoking`, `Air Pollution`, `Coughing of Blood`, etc.? ¿Hay un **orden** claro Low→Medium→High en los factores?
3. **Nombres** interpretables (p. ej. "bajo riesgo/poca exposición", "alta exposición y sintomático"…) y **tamaño**.
4. **Lectura crítica (clave).** Discute la **validez** del dataset: la separación tan nítida y el gradiente perfecto sugieren datos **sintéticos o sobre-estructurados**; ¿qué implica eso para extrapolar a pacientes reales? ¿Qué harías con datos clínicos auténticos (más ruido, solapamiento)?

# <font color="steelblue">Fase 8 — Aplicación interactiva (widget en Colab)</font>

Aplicación **dentro del cuaderno** con **`ipywidgets`** (no despliegue). Implementa **al menos una**:

1. **Explorador de grupos:** desplegable → perfil medio (barras/radar), tamaño, **nivel dominante** y pureza.
2. **Asignador:** *sliders* con los factores principales (`Smoking`, `Air Pollution`, `Coughing of Blood`, `Age`…) → escala → (PCA) → **asigna el cluster** y muestra el **nivel probable** de severidad.
3. **(Opcional)** PC1–PC2 con conmutador "colorear por cluster / por `Level` real" (`plotly`).

> **Aviso:** herramienta **educativa**; modelo no clínico, sobre datos de validez cuestionable.
> El widget solo **usa** el modelo ya ajustado; no reentrena en cada interacción.

# <font color="steelblue">Entregables y formato</font>

* **Cuaderno** ejecutable de principio a fin, con código y **celdas de texto** que justifiquen cada decisión (escalado, nº de grupos, **interpretación de la comparación y escepticismo**).
* **Memoria breve:** EDA, reducción de dimensión, comparación de algoritmos, validación interna, **comparación con la etiqueta ordinal (contingencia + ARI/NMI + `classification_report` + QWK/MAE/Spearman)**, caracterización, **lectura crítica** y la aplicación.
* **Widget(s)** funcionando dentro del cuaderno.
* **Reproducibilidad:** `random_state` fijado y comentario de estabilidad.

## <font color="steelblue">Rúbrica de evaluación (100 pts)</font>

| Fase | Qué se valora | Puntos |
|---|---|---:|
| 1. EDA | Level ordinal, escalas (Age), correlaciones, color por Level | 12 |
| 2. Preprocesado | **Reservar `Level`**, escalado, redundancia | 12 |
| 3. Reducción de dimensión | PCA (interpretación) + MDS/t-SNE coloreado por Level | 14 |
| 4. Clustering y combinación | ≥3 algoritmos, PCA vs original | 12 |
| 5. Validación interna | Codo/silueta, índices, estabilidad, **¿k==3?** | 10 |
| 6. **Comparación con la etiqueta ordinal** | Contingencia + ARI/NMI + mapeo + `classification_report` + **QWK/MAE/Spearman** | 18 |
| 7. Caracterización y **lectura crítica** | Perfiles, gradiente coherente, nombres, **escepticismo (¿sintético?)** | 16 |
| 8. Aplicación (widget) | `ipywidgets` funcional | 6 |
| — Extra | UMAP, consenso, "¿cuántas variables bastan para separar?", comparación con la solución supervisada | +10 |

> **Penalizaciones:** **usar `Level` en el clustering**, **no escalar** (con `Age` dominando), evaluar la etiqueta ordinal **solo con métricas nominales** (sin QWK/MAE), o **celebrar una separación perfecta** sin cuestionar la validez de los datos.

# <font color="steelblue">Pistas y errores típicos</font>

* **Reserva `Level`.** El código original lo borra; aquí se **conserva** como etiqueta y **no** entra en el clustering.
* **Escala por `Age`.** Sin escalar, la edad (rango grande) domina las distancias frente a las escalas 1–9.
* **Etiqueta ORDINAL.** Acompaña ARI/NMI/F1 con **QWK y MAE**: confundir Low↔High es peor que Medium↔High, y solo las ordinales lo reflejan.
* **Demasiado perfecto = sospechoso.** Un ARI/QWK ≈ 1 aquí probablemente dice más del **dataset** (posible síntesis) que de tu método; cruza con el ~100 % supervisado.
* **Gradiente coherente.** Comprueba si los factores crecen ordenadamente con la severidad; en datos reales esperarías más solapamiento.
* **Widget, no despliegue:** `ipywidgets` dentro de Colab.

# <font color="steelblue">Referencias</font>

* The Devastator (2022). *Cancer Patients and Air Pollution: A New Link*. Kaggle.
* *Performance of machine learning algorithms for lung cancer prediction*. Scientific Reports, 2024.
* OMS (2024). *Cancer: Key Facts*.
* Cuadernos del curso: *Componentes principales y variantes*, *Escalas multidimensionales (MDS)*, *Métodos de manifold*, *Clustering (K-Means, jerárquico, DBSCAN, mixturas gaussianas)*.
